# Privacy & Federated Learning

Companion notebook for the [Privacy & Federated Learning lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/15-privacy-and-federated-learning).

We implement the two core mechanisms: **DP-SGD's clip-then-noise** update (and watch the
privacy–utility trade-off), and **Federated Averaging** (train across clients without sharing raw
data). Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1 — DP-SGD: clip per-example gradients, then add noise

Clipping bounds how much any single example can move the model (its sensitivity); the calibrated
Gaussian noise then masks any individual's contribution. We fit a simple linear model both ways.

In [ ]:
# tiny regression problem: y = w*x
Xd = rng.normal(size=200); w_true = 2.0
yd = w_true * Xd + rng.normal(0, 0.1, 200)

def clip(g, C):
    norm = abs(g)
    return g * min(1.0, C / (norm + 1e-12))

def dp_sgd_step(w, Xb, yb, lr, C, sigma):
    grads = [-(yi - w * xi) * xi for xi, yi in zip(Xb, yb)]   # per-example gradients
    clipped = np.array([clip(g, C) for g in grads])           # bound each one's influence
    noisy = clipped.sum() + rng.normal(0, sigma * C)          # add Gaussian noise to the sum
    return w - lr * noisy / len(Xb)

def train(sigma, C=1.0, lr=0.05, epochs=60):
    w = 0.0
    for _ in range(epochs):
        idx = rng.choice(len(Xd), 32, replace=False)
        w = dp_sgd_step(w, Xd[idx], yd[idx], lr, C, sigma)
    return w

print(f'no privacy   (sigma=0):   w = {train(0.0):.3f}  (true = {w_true})')
print(f'some privacy (sigma=1):   w = {train(1.0):.3f}')
print(f'more privacy (sigma=4):   w = {train(4.0):.3f}  (noisier -> less accurate)')

## 2 — The privacy–utility trade-off

More noise (≈ stronger privacy, smaller ε) means a worse model. We sweep the noise multiplier and
measure the error in the learned weight.

In [ ]:
for sigma in [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]:
    errs = [abs(train(sigma) - w_true) for _ in range(20)]
    print(f'noise sigma={sigma:4.1f}  ->  |w_error| = {np.mean(errs):.3f}  (more privacy = more error)')

## 3 — Federated Averaging (FedAvg)

Each client trains on its OWN data and sends back only weights; the server averages them (weighted
by data count). The raw data never leaves the client. We confirm the federated model approaches the
centralized one.

In [ ]:
# split the data across 5 clients (their data stays local)
clients = np.array_split(np.arange(len(Xd)), 5)

def local_train(w, idx, lr=0.05, steps=20):
    for _ in range(steps):
        g = -np.mean((yd[idx] - w * Xd[idx]) * Xd[idx])
        w -= lr * g
    return w

def fedavg(rounds=15):
    w = 0.0
    for _ in range(rounds):
        updates = [local_train(w, c) for c in clients]            # train locally
        counts = np.array([len(c) for c in clients])
        w = np.average(updates, weights=counts)                  # aggregate weights only
    return w

centralized = local_train(0.0, np.arange(len(Xd)), steps=300)
print(f'federated (FedAvg) weight:  {fedavg():.3f}')
print(f'centralized weight:         {centralized:.3f}')
print('FedAvg reaches the centralized solution without any client sharing its raw data.')

## ✏️ Your turn

**Exercise.** Implement `clip_gradient(g, C)` (scale g down so its magnitude is at most C, leave it
unchanged otherwise) and `fedavg_aggregate(weights, counts)` (the data-count-weighted average of the
clients' weights). These are the cores of DP-SGD and federated learning.

In [ ]:
def clip_gradient(g, C):
    # TODO(you): if |g| > C scale it to norm C, else leave unchanged (works for scalar or vector g)
    return ...

def fedavg_aggregate(weights, counts):
    # TODO(you): weighted average of client weights, weighted by their data counts
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(clip_gradient(5.0, 1.0), 1.0)            # large gradient clipped to C
assert np.isclose(clip_gradient(0.3, 1.0), 0.3)           # small gradient untouched
assert np.isclose(np.linalg.norm(clip_gradient(np.array([3.0, 4.0]), 1.0)), 1.0)  # vector -> norm C
# FedAvg weights by data count: a client with more data pulls the average toward it
assert np.isclose(fedavg_aggregate([1.0, 3.0], [10, 30]), (1.0*10 + 3.0*30)/40)
print('\u2713 gradient clipping and FedAvg aggregation are correct')

<details>
<summary>Solution</summary>

```python
def clip_gradient(g, C):
    norm = np.linalg.norm(g)
    return g * min(1.0, C / (norm + 1e-12))

def fedavg_aggregate(weights, counts):
    return np.average(weights, weights=counts, axis=0)
```

Clipping bounds each example's influence (the sensitivity DP noise is calibrated to); FedAvg keeps
raw data on-device and shares only weights. Combined with secure aggregation and DP noise, they form
a layered privacy defense.

</details>